# Découpage du nuage de points Lidar selon l'emprise du projet

Ce notebook vérifie que `processing.lidar.clip.clip_las_to_boundary` découpe correctement un nuage de points (`filters.crop` PDAL) selon une limite + buffer, sur le nuage fusionné produit par le notebook 15.

Ruy-Montceau nécessite en réalité 39 dalles Lidar HD pour une couverture complète ; seule une dalle est actuellement téléchargée (`LHD_FXX_0886_6500`), et elle se trouve dans le coin du rectangle englobant de la commune mais **hors de la forme réelle de la commune**. Le découpage est donc vérifié en deux temps : d'abord avec la vraie limite communale (résultat attendu : 0 point, cas légitime), puis avec un sous-polygone réel à l'intérieur de l'emprise de la dalle, pour prouver quantitativement que `filters.crop` conserve exactement les points attendus.

In [ ]:
import json
import sys
from pathlib import Path

# GeoDataEngine/ (racine du package core/) et son parent (racine du repo, contient data/)
project_root = Path.cwd().parent
repo_root = project_root.parent
sys.path.insert(0, str(project_root))

import geopandas as gpd
import pdal
from shapely.geometry import box

from core.config import load_entities
from download.limites_administratives import fetch_commune_boundary
from processing.lidar.clip import clip_las_to_boundary

merged_path = repo_root / "data" / "raw" / "raster" / "lidar" / "nuage_points.las"

## 1. Découpage selon la vraie limite communale

In [ ]:
csv_path = repo_root / "data" / "configuration" / "entites_exemple.csv"
entity = load_entities(csv_path)[0]
boundary = fetch_commune_boundary(entity.code_insee)

output_real = repo_root / "data" / "processed" / "raster" / "lidar" / "_test_commune.las"
result_real = clip_las_to_boundary(merged_path, boundary, buffer_distance=50, output_path=output_real)

pipeline = pdal.Pipeline(json.dumps([{"type": "readers.las", "filename": str(result_real)}]))
count_real = pipeline.execute()
print(f"Points conserves avec la vraie limite communale : {count_real}")
print("-> attendu : 0, la dalle disponible est hors de la forme reelle de la commune (voir intro).")

result_real.unlink()  # fichier de demonstration, non conserve

## 2. Découpage selon un sous-polygone réel à l'intérieur de la dalle

Validation quantitative : le sous-polygone fait 400 × 400 m avec un buffer de 20 m (soit 440 × 440 m attendus), à l'intérieur de l'emprise de la dalle téléchargée (886000–887000 / 6499000–6500000).

In [ ]:
sub_boundary = gpd.GeoDataFrame(
    {"id": [1]}, geometry=[box(886300, 6499300, 886700, 6499700)], crs="EPSG:2154"
)

output_path = repo_root / "data" / "processed" / "raster" / "lidar" / "nuage_points_test.las"
result = clip_las_to_boundary(merged_path, sub_boundary, buffer_distance=20, output_path=output_path)

pipeline = pdal.Pipeline(json.dumps([{"type": "readers.las", "filename": str(result)}]))
count = pipeline.execute()
arr = pipeline.arrays[0]

expected_bounds = sub_boundary.geometry.buffer(20).total_bounds
print(f"Points apres decoupage : {count}")
print(f"Bornes obtenues  : X [{arr['X'].min()}, {arr['X'].max()}], Y [{arr['Y'].min()}, {arr['Y'].max()}]")
print(f"Bornes attendues : X [{expected_bounds[0]}, {expected_bounds[2]}], Y [{expected_bounds[1]}, {expected_bounds[3]}]")

pipeline_full = pdal.Pipeline(json.dumps([{"type": "readers.las", "filename": str(merged_path)}]))
count_full = pipeline_full.execute()
ratio_attendu = ((440 / 1000) ** 2)
print(f"\nPoints nuage complet : {count_full}")
print(f"Ratio surface (440x440m / 1000x1000m) : {ratio_attendu:.3f}")
print(f"Ratio points obtenu : {count / count_full:.3f}")

## 3. Aperçu du découpage (vue en plan)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

pipeline_full2 = pdal.Pipeline(json.dumps([{"type": "readers.las", "filename": str(merged_path)}]))
pipeline_full2.execute()
arr_full = pipeline_full2.arrays[0]

rng = np.random.default_rng(0)
sample_idx = rng.choice(len(arr_full), size=min(200_000, len(arr_full)), replace=False)
sample = arr_full[sample_idx]

fig, ax = plt.subplots(figsize=(8, 8))
ax.scatter(sample["X"], sample["Y"], s=0.3, color="lightgray", label="Nuage complet (dalle)")
ax.scatter(arr["X"], arr["Y"], s=0.3, color="crimson", label="Points conserves apres decoupage")
ax.set_aspect("equal")
ax.set_title("Decoupage du nuage de points selon un sous-polygone reel")
ax.legend(markerscale=20, loc="upper left", bbox_to_anchor=(1, 1))
plt.show()

output_path.unlink()  # fichier de demonstration, non conserve